FLASK AVANZATO: INTEGRAZIONE DI LLM API

Se abbimo un applicazione che deve servire tanti utenti in modo fluido.
Trasformiamo l'ai in una fabbrica di pensiero.

Flask è un framework web per Python. Permette di eseguire una funzione Python quando un browser o un altro programma invia una richiesta HTTP. La funzione Python fa il suo lavoro (esempio legge le giacenze da SQL) e Flask rimando il risultato al chiamante. Questo è il backend

- Architettura e Factory e Blueprint: struttura modulare
- Gestione Task Asincroni con Celery: l'AI è lenta quindi disaccoppiamo i processi lunghi per la scalabilità (con Celery)
- Straming dei Token: implementazione di Server-Sent Events per interfacce reattive. Per dare all'utente una sensazione di tempo reale

Come evitiamo che il nostro server Flask imploda su se stesso?
Architettura Enterprise in Flask
Oltre il file unico app.py
Un singolo file .py non basta, inizialmente è comodo, ma appena si aggiunge autenticazione, logiche di database, diversi proveder di modelli, quel file diventa un groviglio. In un'architettura Enterprise, dobbiamo smettere di pensare a Flask come a uno script ed iniziare a pensarlo come un sistema.
Dobbiamo separare chi riceve la richiesta (la porta del server), da chi ragione sul dato
Nelle applicazioni reali, la logica di business e le chiamate ai modelli AI diventano rapidamente complesse. Utilizzare un singolo file porta a debiti tecnici insostenibili e difficoltà nel testing.
L'addozione di pattern 'Application Factory' permette di istanziare l'applicazoine in modo dinamico, facilitando la configurazione di diversi ambienti come testing, staging e produzione.

Come si disegna questo sistema modulare?
Immagina di progettare una casa, non metti in cucina il letto, in flask le stanze sono Blueprints. Blueprint per la gestione della chat, uno per la gestione utenti, uno per l'amministrazione, ecc.
La Facotory è invece il campo cantiere, una funzione Python che incaspula la creazione dell'oggetto Flask, permettendo di passare configurazioni diverse a runtime.

Ma oltre all'architettura, come mantieniamo il codice pulito e sicuro?
Organizzazione del Codice
Un'orgamizzazione professionale, richiede 3 pilastri:
- Dependency Injection: non creiamo l'app API dentro ogni funzione, ma la inizializziamo nella factori e la passimo dove serve.
- Centralized Error Handling: gestione centralizzata degli errori. L'utente non deve vedere un errore python brutto, ma un messaggio json educato e standarizzato.
- Configuration Management: mai scrivere l'API Key nel codice, usiamo classi Python per gestire 'API_KEY' e parametri del modello, evitando di cablare valori sensibili direttamente nel codice.

Ma perchè tutta questa fatica architettonica?
Per la Testabilità e la Manutenibilità
Vantaggi del Disaccoppiamento.
Testabilità e Manutenibilità
Separando i 'Service' dalle 'Route', possiamo scrivere unit test per la logica dell'AI senza dover avviare un server web completo. Posso testare il cello AI in isolamento. 
L'indipendenza dai moduli garantisce che un aggiornamento della libreria dell'LLM non richieda modifiche ala logica di routing dell'applicazione.

Ora che abbiamo la struttura affrontiamo il nemico n. 1 , la lentezza.

Le chiamate LLM sono intrinsecamente lente, chiedere il riassunto di un documento lungo, richiede decini di secondi, specialmente per documenti lunghi. 
In una normale connessione http, l'utente fisserebbe una pagaina bianca, spetando non vada errore.
In HTTP, una richiesta così lunga è soggetta a timeout, e degrada l'esperienza dell'utente.
Celery agisce come un sistema di gestione codce che permette di delegare il lavoro pesante a processi worker separati, restituendo immediamtamente una risposta al client.

Celery è lo strumento che gestisce tutto questo.
Task Asincroni con Celery
Gestire l'attesa senza bloccare l'utente.
Le operazione con gli LLM possono richieere decine di sencodi, specialmente con documenti lunghi. In HTTP, una richiesta così lunga è soggetta a timeout e degrada l'esperienza utente.
Celery agisce come un sistema di gestione code che permette di delegare il lavoro pesante a processi worker separati, restituendo immediatamente una risposta al client.

Come si coordinano questi lavoratori in background
Architettura Distributed Task Queue
L'architettura asincrona si basa su 3 attori: Worker, Broker e Result Backend
- Broker: un intermediario come Redis o RabbitMQ che riceve i messaggi dei task e li smista ai worker disponibili. Come per esempio Redis, è l'ufficio posta che riceve i messaggi
- Worker: processi indipendenti che eseguono la logica Python (es. la chiamata API all''LLM) in backgrouind. Sono gli operai specializzati che leggono i messaggi e fanno il lavoro duro sulla GPU o chiamano un API esterna.
- TaskID: identificativo univoco restituito istantaneamente all'utente per monitorare lo stato dell'operazione asincrona
- Il throughput del sistema aumenta distribuendo il carico su 'N' worker indipendenti dalla risposta HTTP.
Se abbiamo troppe richieste aumentiamo gli operai, senza mai rallentare il caricamento della pagina web all'utente finale.

Ma come facciamo a dire all'utente che il lavoro è finito?
Abbiamo diverse strategie

- Pooling: il client chiede, ogni 2 secondi, 'è pronto?'
- Task Retries: gestione automatica degli errori se l'API fallisce Celery riprova l'operazione con un ritardo crescente, senza che l'utente debba intervenire.
Fondamentale è la persistenza, una volta che l'AI ha finito di generare un testo, lo salviamo in un database, affinchè sia recuperabile in qualsiasi  moenti, anche giorni dopo, anche dopo la chiusura del worker.

Gestire centinaia di utenti dobbiamo organizzare bene le code di lavoro.
Ottimizzazione delle Code
Isolamento delle risorse.
Un errore comune è mettere tutti i task nella stessa coda, immagina se le richieste di "ciao" veloci, finissero dietro il riassunto di un libro di 500 pagine, estremamente lento, dobbiamo isolare le risorse. 
Creiamo una corsia preferenziale per i task iterattivi brevi ed una corsia per i mezzi pesanti.
Questo garantisce che l'app sia sempre scattante.

Monitorando queste corsia poi capiamo se dobbiamo aggiungere nuove istanze o se l'architettura sta reggendo bene il carico.

C'è un modo più moderno per gestire i carichi, non aspettare affatto ed inviare i dati un pezzo alla volta (streaming)

Streaming con Server-Sent Events
Reattività in tempo reale stile ChatGPT
ChatGPT fa apparire la parole una ad una, da un senso di interittività immediata. 
L'attesa della risposta completa di un LLM può sembrare un'eternità. Lo streaming permette di mostrare all'utente i token man mano che vengono generati dal modello.
Utilizziamo i generatori Python e il protocollo HTTP SSE (leggero ed unidirezionale, dal server verso il browser) per mantenere una connessione aperta che invia dati incrementali senza la complessità dei webSocket

Implementazione dello Streaming
Il cuore dello streaming sono i generatori
Usiamo la parola chiave Yield per immetere dati senza chiudere la funzione
La latenza che utente percepisce non è più il tempo totale dell'elaborazione, ma il tempo per ricevere il primo token, solitamente pochi secondi.
- stream=True: parametro fondamentale nelle API degli LLM per ricevere una risposta frammentata in pezzi (chuncks)
- Python Generators: funzioni che usano 'yield' per emettere dati sequenzialmente invece di restituire una lista finita.
- SSE Protocol: un protocollo unidirezionale leggero che invia messaggi testuali preceduti dal prefisso data: tramite Flask Response
- La lataenza percepita dall'utente si riduce drasticamente, avvicinandosi al tempo di generazione del primo token.
Cambia la psicologia dell'attesa
Ma come facciamo a dire al browser che deve aspettarsi un flusso e non un dato intero?

Gestione della Connessione
Dobbiamo configurare correttamente gli header HTTP in Flask affinchè il browser non attenda la fine della richiesta per processare i dati, per questo usiamo MIME Type text/event-stream (eh non chiudere la connessione stanno arrivando altri pezzi)
Dobbiamo poi processare i chunk JSON che API ha generato, estraendo solo il testo utile e scartando tutti i metadati tecnici
Lato frontend, Javascript userà l'API 'EventSource' per leggero lo stream di byte e aggiornare il DOM (chat) in tempo reale, riga dopo riga.

Contronto SSE vs WebSockets
Scegliere lo strumento giusto
SSE è ideale per la solo ricezione di testo (unidirezionale), è più semplice da implementare su HHTP standard e gestisce nativamente la connessione.
Per applicazioni AI di sola generazione di testo, SSE rappresenta il miglior compromesso tra prestazioni e semplicità architetturale.

VEDI CARTELLA 09_FLASK PER ESEMPIO